In [ ]:
from __future__ import annotations
import datetime as dt
from typing import NamedTuple
from collections.abc import Iterable
from random import random
import sys
from contextlib import contextmanager

from tuplesaver.engine import Engine, RecordNotFoundError
from tuplesaver.model import Row, TableRow

In [ ]:
from collections.abc import Iterator
import apsw
import apsw.ext

class ListType(TableRow):
    codename: str


class List(TableRow):
    name: str  # unique
    listtype: ListType

class ListSection(TableRow):
    title: str
    list: List


class ListItem(TableRow):
    txt: str
    listsection: ListSection


@contextmanager
def engine() -> Iterator[Engine]:
    conn = apsw.Connection("xxx.db")
    with apsw.ext.Trace(sys.stdout, conn):
        conn.pragma("journal_mode", "WAL")

        with conn:
            e = Engine(conn)

            e.ensure_table_created(ListType)
            e.ensure_table_created(List)
            e.ensure_table_created(ListSection)
            e.ensure_table_created(ListItem)

        with e.connection:
            yield e

with engine() as e:
    shopping = e.insert(ListType("shopping"))
    groceries = e.insert(List("Grocery", shopping))

# Example using relational filters and lazy loading
Uses `Engine.insert`, `Engine.find`, and `Engine.select` with expression filters.

In [ ]:
# populate the database with some data
with engine() as e:
    section_items = {
        'produce': [
            'Other fruit',
            'Berries',
            'Limes',
            'oranges, not the very large ones, but not the small ones either. Also they should be extra sweet, and heavy (and therefore JUICY 💦 yum!)',
        ],
        'deli': ['sliced cheese', 'Lunch meat', 'Goat cheese'],
        'meat': ['chicken'],
        'dairy': ['Milk', 'Eggs', 'Sour cream', 'Cheddar block'],
        'frozen': ['Frozen Veggies'],
        'drinks': ['sparkling water', 'Pop'],
        'snacks': ['peanuts', 'Triscut', 'Goldfish'],
        'ingredient': ['tortillas', 'Peanut butter', 'Prescription'],
        'etc': ['cat food', 'Tide'],
    }

    for section, items in section_items.items():
        section = e.insert(ListSection(section, groceries))
        for item in items:
            e.insert(ListItem(item, section))


In [ ]:
list_name = "Grocery"
section_items = {}

with engine() as e:
    try:
        mylist = e.find(List, List.name == list_name)
    except RecordNotFoundError:
        mylist = None

if mylist is None:
    print(f"List {list_name} not found")
    listitems = []
else:
    with engine() as e:
        listitems = e.select(ListItem, ListItem.listsection.list.name == list_name)

    if len(listitems) == 0:
        print(f"No items found for list {list_name}")

    for item in listitems:
        print(item)
        section_items.setdefault(item.listsection.title, []).append(item)
        print(f"Added item {item.txt} to section {item.listsection.title}")

    print(f"Rendering template for list {mylist}")